In [1]:
# ===============================
# Helcim Take Home Technical Challenge
# Interview 4: SQL Preparation
# ===============================

# Author: Shelly Resurreccion
# Purpose: Practice writing SQL queries that may come up based on the dataset provided.

---
### 1.0 Set Up

In [2]:
# ===============================
# Imports:
# ===============================
# --- Standard library ---
import sys
import os

# --- Data handling ---
import sqlite3
import pandas as pd

In [3]:
# ===============================
# Functions:
# ===============================
# N/A

In [4]:
# ===============================
# Load Data:
# ===============================
# Notes: The dataset was encoded in Latin-1 rather than UTF-8, so the encoding was explicitly specified when loading the CSV.
df = pd.read_csv(
    "../data/US_Accidents_March23.csv",
    encoding="latin1"
)

conn = sqlite3.connect(":memory:")
US_Accidents = df.to_sql("US_Accidents", conn, index=False, if_exists="replace")

In [5]:
df.columns

Index(['Unnamed: 0', 'ID', 'Source', 'Severity', 'Start_Time', 'End_Time',
       'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
       'Description', 'Street', 'City', 'County', 'State', 'Zipcode',
       'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp',
       'Temperature_Range(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
       'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
       'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing',
       'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station',
       'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop',
       'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight'],
      dtype='object')

In [6]:
# Question 1
# Top 5 states with the most severe accidents

query = """
SELECT
    State,
    COUNT(*) AS Accident_Count
FROM US_Accidents
WHERE Severity >= 3
GROUP BY State
ORDER BY Accident_Count DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,State,Accident_Count
0,VA,1153
1,NC,800
2,GA,749
3,PA,681
4,CO,469


In [7]:
# Question 2
# Accidents by hour + severity

query = """
SELECT
    strftime('%H', Start_Time) AS Hour,
    Severity,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Hour, Severity
ORDER BY Accident_Count DESC;
"""

pd.read_sql(query, conn)

,Hour,Severity,Accident_Count
0,15,2,17915
1,16,2,17568
2,07,2,16787
3,17,2,16549
4,14,2,15474
5,08,2,14867
6,18,2,13415
7,06,2,12953
8,13,2,12810
9,12,2,11372


In [8]:
# Question 3
# Weather conditions associated with highest severity

query = """
SELECT
    Weather_Condition,
    AVG(Severity) as AVG_Severity
FROM US_Accidents
GROUP BY Weather_Condition
ORDER BY AVG_Severity DESC;
"""

pd.read_sql(query, conn)

,Weather_Condition,AVG_Severity
0,Light Rain Shower,2.200000
1,Blowing Snow / Windy,2.166667
2,Blowing Dust / Windy,2.133333
3,Light Freezing Rain,2.120805
4,Heavy Snow / Windy,2.101266
...,...,...
76,Freezing Drizzle,2.000000
77,Drizzle and Fog,2.000000
78,Drizzle / Windy,2.000000
79,Blowing Snow,2.000000


In [9]:
# Question 4
# Rush hour vs off-peak accident comparison

query = """
SELECT
    CASE
        WHEN CAST(strftime('%H', Start_Time) AS Integer) BETWEEN 7 AND 9
            OR CAST(strftime('%H', Start_Time) AS Integer) BETWEEN 16 AND 18
        THEN 'Rush Hour' 
        ELSE 'Off Peak' 
    END AS Time_Category,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Time_Category;
"""

pd.read_sql(query, conn)

,Time_Category,Accident_Count
0,Off Peak,154245
1,Rush Hour,92388


In [10]:
# Question 5
# Month-over-month accident trend

query = """
SELECT
    strftime('%Y', Start_Time) AS Year,
    strftime('%m', Start_Time) AS Month,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Year, Month
ORDER BY Year,Month DESC;
"""

pd.read_sql(query, conn)

,Year,Month,Accident_Count
0,2023,03,30187
1,2023,02,55532
2,2023,01,160914


In [11]:
# Question 6
# Accidents in California in January 2023

query = """
SELECT 
    COUNT(*)
FROM US_Accidents
WHERE State = 'CA'
    AND strftime('%Y-%m-%d', Start_Time) >= '2023-01-01'
    AND strftime('%Y-%m-%d', Start_Time) < '2023-01-31';
"""

pd.read_sql(query, conn)

,COUNT(*)
0,34618


In [12]:
# Question 7
# Accidents by day of the week

query = """
SELECT
    strftime('%w', Start_Time) AS Day_Of_Week,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY Day_Of_Week
ORDER BY Day_Of_Week;
"""

pd.read_sql(query, conn)

,Day_Of_Week,Accident_Count
0,0,27727
1,1,36863
2,2,41421
3,3,39547
4,4,38174
5,5,39873
6,6,23028


In [13]:
# Question 8
# States with more than 10,000 severe accidents

query = """
SELECT
    State,
    COUNT(*) AS Accident_Count
FROM US_Accidents
GROUP BY State
HAVING Accident_Count >= 10000
ORDER BY Accident_Count DESC;
"""

pd.read_sql(query, conn)

,State,Accident_Count
0,CA,74559
1,FL,22243
2,TX,13824
3,VA,11426
4,NY,10820
5,PA,10605


In [14]:
# Question 9
# Average severity by weather condition (excluding NULLs)

query = """
SELECT
    Weather_Condition, 
    AVG(Severity) AS AVG_Severity
FROM US_Accidents
WHERE Weather_Condition IS NOT NULL
GROUP BY Weather_Condition
ORDER BY AVG_Severity DESC;
"""

pd.read_sql(query, conn)

,Weather_Condition,AVG_Severity
0,Light Rain Shower,2.200000
1,Blowing Snow / Windy,2.166667
2,Blowing Dust / Windy,2.133333
3,Light Freezing Rain,2.120805
4,Heavy Snow / Windy,2.101266
...,...,...
75,Freezing Drizzle,2.000000
76,Drizzle and Fog,2.000000
77,Drizzle / Windy,2.000000
78,Blowing Snow,2.000000


In [15]:
# Question 10
# Top 3 states by number of accidents per month

query = """
WITH monthly_count AS (
    SELECT
        State,
        strftime('%m', Start_Time) AS Month,
        COUNT(*) AS Accident_Count
    FROM US_Accidents
    GROUP BY State, Month
)

SELECT 
    State,
    AVG(Accident_Count) AS AVG_Accident_Count
FROM monthly_count
GROUP BY State
ORDER BY AVG_Accident_Count DESC
LIMIT 3;
"""

pd.read_sql(query, conn)

,State,AVG_Accident_Count
0,CA,24853.000000
1,FL,7414.333333
2,TX,4608.000000


In [16]:
# Question 11
# Show all states in the top 25% of accidents

query = """
WITH state_accidents AS (
    SELECT
        State, 
        COUNT(*) AS Accident_Count
    FROM US_Accidents
    GROUP BY State 
),

state_ranked AS (
    SELECT *,
        PERCENT_RANK () OVER (ORDER BY Accident_Count) AS PCT_Rank
    FROM state_accidents
)

SELECT
    State,
    Accident_Count,
    PCT_Rank
FROM state_ranked
ORDER BY PCT_Rank DESC;
"""

pd.read_sql(query, conn)

,State,Accident_Count,PCT_Rank
0,CA,74559,1.000000
1,FL,22243,0.977273
2,TX,13824,0.954545
3,VA,11426,0.931818
4,NY,10820,0.909091
5,PA,10605,0.886364
6,NC,9133,0.863636
7,MN,8949,0.840909
8,SC,8534,0.818182
9,GA,8457,0.795455


In [18]:
# Question 12
# How does each city compare to the median number of accidents?

query = """
WITH city_counts AS (
    SELECT
        City,
        COUNT(*) AS Accident_Count
    FROM US_Accidents
    GROUP BY City
),

avg_count AS (
    SELECT
        AVG(Accident_Count) AS AVG_Accident_Count
    FROM city_counts
)

SELECT
    c.City,
    c.Accident_Count,
    a.AVG_Accident_Count,
    c.Accident_Count - a.AVG_Accident_Count AS Diff_Against_AVG
FROM city_counts c
CROSS JOIN avg_count a
ORDER BY Diff_Against_AVG DESC;
"""

pd.read_sql(query, conn)

,City,Accident_Count,AVG_Accident_Count,Diff_Against_AVG
0,Los Angeles,5880,37.115576,5842.884424
1,Miami,5238,37.115576,5200.884424
2,Dallas,3261,37.115576,3223.884424
3,Atlanta,3182,37.115576,3144.884424
4,San Diego,2759,37.115576,2721.884424
...,...,...,...,...
6640,Zenia,1,37.115576,-36.115576
6641,Zieglerville,1,37.115576,-36.115576
6642,Zortman,1,37.115576,-36.115576
6643,Zuni,1,37.115576,-36.115576


In [ ]:
# Question 13
# Median in SQLite

query = """
"""

pd.read_sql(query, conn)

In [ ]:
# Question 14
# Distance between two points (classic interview question)

query = """
"""

pd.read_sql(query, conn)

In [ ]:
# Question 15
# Compute the duration of the accident

query = """
"""

pd.read_sql(query, conn)

In [ ]:
# Question 16
# Which state had the longest duration of accidents?

query = """
"""

pd.read_sql(query, conn)